# PrivaDE workflow

In this scenario, we assume Bob has multiple data points to contribute to Alice's ML model. Alice is trying to value the dataset as a whole, judging on the diversity, uncertainty of the datasets as well as the current model's performance on the dataset. Moreover, the parties are assumed to be malicious, which means they might deviate from the protocol to maximize their own utility.

## Part 0: Setup

We set up Alice's model and Bob's dataset.

In [24]:
import os
import torch
import sys
import random
sys.path.append('..')  # Add privade directory to path
from privade.data import get_dataset
from privade.models import get_model
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N = 1000 #Bob's dataset size

full_model = get_model('lenet5', 'mnist')
full_data = get_dataset('mnist')

# Randomly select 1000 images as Bob's dataset
indices = random.sample(range(len(full_data)), N)
bob_images = np.array([full_data[i][0].numpy() for i in indices])
bob_labels = np.array([full_data[i][1]  for i in indices])

# Ramdomly select 1000 images as Alice's initial dataset
indices = random.sample(range(len(full_data)), N)
alice_images = np.array([full_data[i][0].numpy() for i in indices])
alice_labels = np.array([full_data[i][1]  for i in indices])

# Train the model for a few epochs
alice_images_tensor = torch.FloatTensor(alice_images)
alice_labels_tensor = torch.LongTensor(alice_labels)
alice_dataset = TensorDataset(alice_images_tensor, alice_labels_tensor)
alice_dataloader = DataLoader(alice_dataset, batch_size=32, shuffle=True)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(full_model.parameters(), lr=0.001)
full_model.train()
for epoch in range(5):
    running_loss = 0.0
    for images, labels in alice_dataloader:
        optimizer.zero_grad()
        outputs = full_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}/5, Loss: {running_loss/len(alice_dataloader):.4f}')
    
# Save models and datasets
torch.save(bob_images, 'data/bob_images.pth')
torch.save(bob_labels, 'data/bob_labels.pth')
torch.save(full_model.state_dict(), 'data/alice_full_model.pth')

Epoch 1/5, Loss: 1.7458
Epoch 2/5, Loss: 0.7711
Epoch 3/5, Loss: 0.5205
Epoch 4/5, Loss: 0.4005
Epoch 5/5, Loss: 0.3473


### Model distillation

For Alice's preprocessing, she has an optional step to do model distillation, to obtain a smaller model for data evaluation.

In [25]:
from privade.distillation import train_distilled_model

student_model = get_model('lenetxs', 'mnist')

kd_train_loader = DataLoader(alice_dataset, batch_size=32, shuffle=True)

trained_student_model = train_distilled_model(full_model, student_model, kd_train_loader, kd_train_loader,epochs=10)

torch.save(trained_student_model.state_dict(), "data/alice_student_model.pth")

Starting knowledge distillation training on cuda
Epochs: 10, LR: 0.01, Alpha: 0.7, Temperature: 4.0
Epoch: [0][0/32] Loss 3.9728 (3.9728) Acc@1 9.375 (9.375)
Epoch: [0][10/32] Loss 4.0228 (4.1644) Acc@1 31.250 (13.352)
Epoch: [0][20/32] Loss 2.1833 (3.8789) Acc@1 43.750 (25.000)
Epoch: [0][30/32] Loss 1.7476 (3.4315) Acc@1 71.875 (34.879)
Epoch [1/10] - Train Loss: 3.4126 (CE: 1.9487, KD: 6.8286) Train Acc: 35.30%
Epoch: [1][0/32] Loss 1.4586 (1.4586) Acc@1 78.125 (78.125)
Epoch: [1][10/32] Loss 1.0681 (1.3238) Acc@1 75.000 (74.432)
Epoch: [1][20/32] Loss 1.2311 (1.1776) Acc@1 71.875 (75.149)
Epoch: [1][30/32] Loss 0.5721 (1.1109) Acc@1 93.750 (75.403)
Epoch [2/10] - Train Loss: 1.1056 (CE: 0.7563, KD: 1.9208) Train Acc: 75.60%
Epoch: [2][0/32] Loss 0.7257 (0.7257) Acc@1 78.125 (78.125)
Epoch: [2][10/32] Loss 0.6433 (0.5267) Acc@1 81.250 (88.920)
Epoch: [2][20/32] Loss 0.4450 (0.4875) Acc@1 90.625 (89.286)
Epoch: [2][30/32] Loss 0.3879 (0.4792) Acc@1 93.750 (89.516)
Epoch [3/10] - Trai

In [26]:
from torchsummary import summary
summary(trained_student_model, input_size=alice_images[0].shape)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [-1, 3, 24, 24]              78
            Square-2            [-1, 3, 24, 24]               0
           Flatten-3                 [-1, 1728]               0
            Linear-4                  [-1, 900]       1,556,100
              ReLU-5                  [-1, 900]               0
            Linear-6                   [-1, 32]          28,832
              ReLU-7                   [-1, 32]               0
            Linear-8                   [-1, 10]             330
Total params: 1,585,340
Trainable params: 1,585,340
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.05
Params size (MB): 6.05
Estimated Total Size (MB): 6.10
----------------------------------------------------------------


### Split Model

Alice also needs to perform split model to split her model $M$ into $A,B,C$. 


In [27]:
from privade.split import split_model
try:
    
    # Split the model
    model_A, model_B, model_C, split_stats = split_model(
        data_loader=alice_dataloader,
        model=trained_student_model,
    )
    model_A.to(device)
    model_B.to(device)
    model_C.to(device)

    print(f"\nSplit successful!")
    print(f"First activation layer: {split_stats['first_activation_layer']}")
    print(f"Optimal boundary layer: {split_stats['optimal_layer']}")
    print(f"Privacy preserved rate: {split_stats['privacy_preserved_rate']:.3f}")
    print(f"Client model (model_B): {len(list(model_B.children()))} layers")
    print(f"Server model (model_C): {len(list(model_C.children()))} layers")
    
except Exception as e:
    print(f"Error during splitting: {e}")
    import traceback
    traceback.print_exc()

Found first activation layer at index 1: Square
First activation layer:  1
Identifying candidate layers for B/C boundary...
Candidates [4, 0]
Found 1 candidate layers: [4]
Starting boundary analysis with DINA attack...
Attack epochs: 20
This may take a while...
Starting boundary layer evaluation...


Evaluating layers:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating layer 4...
    Split layer: 4
    Distillation taps (pre-ReLU conv), ordered near→far: [3]
    Sub-blocks: [[0, 1, 2, 3, 4]]
    Using linear inverse network
    Feature path: [900, 900]
    Final spatial size: (24, 24)
    Loss coefficients: [1.0, 3.0]
    Training DINA attack for 20 epochs...


DINA Epoch 1/20: 100%|██████████| 32/32 [00:00<00:00, 774.08it/s, Loss=24.9346]


Epoch 1: Average Loss = 31.7961


...

Epoch 19: Average Loss = 31.1427


DINA Epoch 20/20: 100%|██████████| 32/32 [00:00<00:00, 677.00it/s, Loss=23.6803]


Epoch 20: Average Loss = 30.9685
    Evaluating DINA attack...


Evaluating layers: 100%|██████████| 1/1 [00:01<00:00,  1.01s/it]

  Privacy Preserved Rate: 0.000

BOUNDARY ANALYSIS RESULTS

Layer 4:
  Privacy Preserve Rate: 0.000
  Attack Success: 1.000
  Avg SSIM: 0.591

OPTIMAL BOUNDARY LAYER: None
Using layer 4 as fallback.

Flattened model has 8 layers:
  0: Conv2d
  1: Square
  2: Flatten
  3: Linear
  4: ReLU
  5: Linear
  6: ReLU
  7: Linear

Three-Model Split Statistics:
Total layers: 8
Model A ends at layer: 1 (Square)
Model B: layers 2 to 4 (3 layers)
Model C: layers 5 to 7 (3 layers)
Privacy Rate: 0.000
Attack Success: 1.000
Average SSIM: 0.591

Model A layers (2):
  0: Conv2d
  1: Square

Model B layers (3):
  0: Flatten
  1: Linear
  2: ReLU

Model C layers (3):
  0: Linear
  1: ReLU
  2: Linear

Split successful!
First activation layer: 1
Optimal boundary layer: 4
Privacy preserved rate: 0.000
Client model (model_B): 3 layers
Server model (model_C): 3 layers


In [28]:
model_A.to(device)
model_A(torch.tensor(alice_images).to(device)).shape

torch.Size([1000, 3, 24, 24])

In [29]:
#Optionally, add a weight mixer to model A and unmix in model_B. For the experiment, we will not add it for now.
from privade.weight_mixer import weight_mixer

model_A, model_B = weight_mixer(model_A, model_B)

In [30]:
#Save all models

torch.save(model_A.state_dict(),"data/model_a.pth")
torch.save(model_B.state_dict(),"data/model_b.pth")
torch.save(model_C.state_dict(),"data/model_c.pth") 

## Part 1: Representative Set selection

Once all models are prepared, Bob can start selecting a representative set from his dataset.

In [31]:
#Optional step: Dimension reduction
from privade.dim_reduction import reduce_image_dimensions

target_dimension = 50
reduced_images, _,_ = reduce_image_dimensions(bob_images, target_dimension)

torch.save(reduced_images, 'data/reduced_images.pth')
torch.save(bob_labels, 'data/reduced_labels.pth')


Original image shape: (1000, 1, 28, 28)
Original feature space: 784
Target dimensions: 50
Flattened shape: (1000, 784)
After random projection: (1000, 50)
After scaling: (1000, 50)


In [32]:
#Perform clustering
from privade.clustering import kmeans_clustering

rep_set_size = 20

representative_set = kmeans_clustering(reduced_images, rep_set_size)

In [33]:
representative_points = reduced_images[representative_set]
rep_points = bob_images[representative_set]
rep_labels = bob_labels[representative_set]
torch.save(rep_points, 'data/rep_points.pth')
torch.save(rep_labels, 'data/rep_labels.pth')
dists = np.linalg.norm(reduced_images[:, None] - representative_points[None, :], axis=2)
min_dists = np.min(dists, axis=1)
max_min_distance = np.ceil(np.max(min_dists))
print("Maximum of the minimum distances:", max_min_distance)

Maximum of the minimum distances: 2.0


Next, we will prepare a setup for performing the challenge protocol with alice.

In [34]:
from privade.challenge_protocol import setup_challenge_protocol

# setup_challenge_protocol()

In [35]:
# from privade.challenge_protocol import create_proof, verify_proof

# M = 20 #challenge number

# #Alice randomly selects M points from the whole dataset
# indices = random.sample(range(N), M)
# print(indices)

# #For each data point do the Challenge Protocol

# for idx in indices:
#     #Find the index from representative_points which has the min distance from the selected point
#     selected_point = reduced_images[idx]
#     dists = np.linalg.norm(representative_points - selected_point, axis=1)
#     min_index = np.argmin(dists)
#     print(np.min(dists), min_index)
#     cp_data = {
#         "messageArray": selected_point.tolist(),
#         "idx": int(min_index),
#         "allPoints": representative_points.tolist(),
#         "d": int(max_min_distance),
#         "r": 0x12345678
#     }
#     assert len(selected_point.tolist()) == 50
#     assert len(representative_points.tolist()) == 20
    
#     proof_file = "proof.json"
    
#     assert create_proof(cp_data,proof_file)
    
#     assert verify_proof(proof_file)
    

## Part 2: Model Inference


The next part of the process involves model inference of A, B and C.

In [36]:
#Model A insecure inference
model_A = model_A.to(device)
model_A_output = model_A(torch.tensor(rep_points).to(device))


In [37]:
# Create a mapping from saved model structure to your simplified net structure
# def create_compatible_state_dict(saved_state_dict):
#     """
#     Convert the saved model_A state_dict to match the simple net structure
#     """
#     compatible_state = {}
    
#     # Mapping from saved keys to net keys
#     key_mapping = {
#         '0.0.weight': '0.weight',      # Conv2d weight
#         '0.0.bias': '0.bias',          # Conv2d bias  
#         '2.linear.weight': '3.weight', # Linear weight (after Flatten at index 2, Linear is at index 3)
#         '2.linear.bias': '3.bias'
#     }
    
#     for saved_key, net_key in key_mapping.items():
#         if saved_key in saved_state_dict:
#             compatible_state[net_key] = saved_state_dict[saved_key]
#             print(f"Mapped {saved_key} -> {net_key}: {saved_state_dict[saved_key].shape}")
    
#     return compatible_state

# # Load the saved weights and create compatible state dict
# saved_state = torch.load("/home/thomas/secure-data-valuation/notebooks/data/model_a.pth", weights_only=True)
# compatible_state = create_compatible_state_dict(saved_state)

# # Save the converted model for future use
# torch.save(compatible_state, "/home/thomas/secure-data-valuation/notebooks/data/model_a_compatible.pth")
# print("Saved compatible model to model_a_compatible.pth")

# class Square(nn.Module):
#     """Elementwise square activation: f(x) = x^2"""
#     def __init__(self, inplace: bool = False):
#         super().__init__()
#         self.inplace = inplace

#     def forward(self, x):
#         return x.mul_(x) if self.inplace else x * x

# net = nn.Sequential(
#     nn.Conv2d(1, 3, kernel_size=(5, 5), stride=(1, 1)) ,
#     Square(),
#     nn.Flatten(),
#     nn.Linear(1728,1728)
# )

# net.load_state_dict(torch.load("/home/thomas/secure-data-valuation/notebooks/data/model_a_compatible.pth"), strict=False)


In [38]:
#MPC 
from privade.mpc_inference import setup, inference

#Update the file

#Setup
# setup("model_a_inf")

# #Perform inference
# inference("model_a_inf")

In [39]:
#MOdel B: Bob inference
from privade.cnczk import choose_random_layers, collect_sequential_activations, get_layer, setup_zkp
import asyncio
#Plaintext inference
model_B.to(device)
model_B_output = model_B(model_A_output)

#Randomly choose layers:
layers = choose_random_layers(model_B,1)
print(layers)

for layer in layers:
    await setup_zkp(model_B, model_A_output, layer, 'pw')


[0]


In [40]:
import random
# Alice randomly chooses s number of points
s = 5
points_to_check = random.sample(range(len(model_A_output)), s)
selected_A_output = model_A_output[points_to_check]
selected_A_output.shape

torch.Size([5, 1728])

In [41]:
from privade.cnczk import prove_zkp, verify_zkp
for layer in layers:
    await prove_zkp(model_B,selected_A_output,layer)
    
for layer in layers:
    await verify_zkp(layer)

In [42]:
#Model C: Alice inference
#This is similar, just with different visibility settings for model weights and data.
model_C.to(device)
model_C_output = model_C(model_B_output)

In [43]:
#Randomly choose layers:
layers = choose_random_layers(model_C,1)
print(layers)

for layer in layers:
    await setup_zkp(model_C, model_B_output, layer, 'pi')

[1]


In [44]:
s = 5
points_to_check = random.sample(range(len(model_B_output)), s)
selected_B_output = model_B_output[points_to_check]
selected_B_output.shape

torch.Size([5, 900])

In [ ]:
for layer in layers:
    await prove_zkp(model_C,selected_B_output,layer)
    
for layer in layers:
    await verify_zkp(layer)


RuntimeError: Failed to generate witness: [graph] [tensor] dimensionality error when manipulating a tensor: Cannot reshape tensor of length 900 to [1, 32]

## Part 3: Secure Scoring

In [ ]:
#Prepare Alice and Bob's private input for MPC
#We use unreduced data for inference but reduced data for the diversity calculation here
points_to_submit = reduced_images[representative_set]
labels_to_submit = rep_labels

Bob_input = (points_to_submit, labels_to_submit)
Alice_input = model_C_output.cpu().detach()


from privade.scoring import prepare_inputs

assert prepare_inputs(Bob_input, Alice_input)

In [ ]:
from privade.scoring import compile_program
assert compile_program()